<a href="https://colab.research.google.com/github/TheGit-Father/cuda-matmul/blob/main/naive_matmul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
from numba import cuda

#cuda kernel
@cuda.jit
def matmul_naive(A,B,C):
  row,col=cuda.grid(2)

  if row<C.shape[0] and col<C.shape[1]:
    tmp=0.0
    for k in range(A.shape[1]):
       tmp+=A[row,k]*B[k,col]

    C[row,col]=tmp

#Matrix size
N=256

#Random Matrices
A=np.random.rand(N,N).astype(np.float32)
B=np.random.rand(N,N).astype(np.float32)

#Output Matrix
C=np.zeros((N,N), dtype=np.float32)

#Copy to GPU
d_A=cuda.to_device(A)
d_B=cuda.to_device(B)
d_C=cuda.device_array((N,N), dtype=np.float32)

#Thread configuration
threads_per_block=(16,16)

blocks_per_grid_x=(N+threads_per_block[0]-1)//threads_per_block[0]
blocks_per_grid_y=(N+threads_per_block[1]-1)//threads_per_block[1]

blocks_per_grid=(blocks_per_grid_x, blocks_per_grid_y)

#Launch kernel
matmul_naive[blocks_per_grid, threads_per_block](d_A,d_B,d_C)

#Copy result back
C=d_C.copy_to_host()

#Numpy refernce
C_numpy=np.matmul(A,B)

#Verify
print("Correct:", np.allclose(C,C_numpy, atol=1e-3))

Correct: True
